# Capstone — The Injaz Experimentation and Impact Evaluation Package
# المشروع الختامي — حزمة التجريب وتقييم الأثر لمنصة إنجاز

**SDA-DSC-213 · Experimentation, A/B Testing and Causal Inference · SDAIA Academy**

*Day 4, Hours 3–5 · 100 points · pass ≥ 70 · distinction ≥ 90*

---

> **This is a SKELETON, not a solution.** Every section below is scaffolded with the
> structure you must produce and a `# TODO` describing exactly what goes there. Fill it in.
> The notebook runs top-to-bottom in its unfinished state — TODOs report themselves rather
> than crashing — so you can execute at any point to see how far you have got.

## Your mandate

You are the experimentation lead for **Injaz (إنجاز)**, the Saudi unified digital-government
services platform. Leadership faces two linked decisions before the next budget cycle:

1. **Launch** the new **AI-assisted document-and-form autofill** feature nationally?
2. **Scale** the **regional reminder-notification programme** piloted region-by-region last
   year without randomisation?

Design and analyse a clean experiment for the feature, produce a credible causal evaluation
of the pilot where randomisation was impossible, and deliver a decision memo leadership can
act on.

Everything you built in Labs 1–7 is a component. The capstone is the **integration**, plus
your own rigour and communication.

## Architecture (target state)

```
Frame ──> Design doc (frozen plan) ──> Power/sizing ──> [A/B: assign → SRM → balance → analyse]
                                                     └─> [Pilot: DAG → identify → estimate → refute]
   A/B effect (CUPED, robust CI) ─┐
   Pilot effect (DiD/matching CI) ┼──> reconcile ──> Decision plot (vs threshold) ──> Decision memo
   guardrails + validation ───────┘                                                    (ship/scale/hold)
Reproducible notebook: fixed seed · frozen plan versioned before results · honest nulls
```

## Milestones

| Milestone | Section here | Due | Gate |
|---|---|---|---|
| **M-A** framed question + frozen design doc + sizing | §1–§3 | Day 4 H3 start | instructor spot-check of the plan |
| **M-B** A/B analysed (SRM, balance, effect, guardrails) | §4–§6 | Day 4 H3 end | integrity checks pass |
| **M-C** pilot evaluated (DiD/matching + validation) | §7–§8 | Day 4 H4 mid | refutation / validation present |
| **M-D** decision memo + extension | §9–§11 | Day 4 H4 end | full self-audit checklist |
| **M-E** presentation + submission | §12 | Day 4 H5 | rubric scoring |

## Grading rubric (100 points)

| Criterion | Weight | Section |
|---|---|---|
| Experiment design & analysis plan | 15 | §2 |
| Power & sample sizing | 15 | §3 |
| Experiment integrity | 10 | §4 |
| A/B analysis correctness | 20 | §5–§6 |
| Causal evaluation | 20 | §7–§8 |
| Decision communication | 15 | §9–§10 |
| Reproducibility & integrity | 5 | throughout + §11 |

**Extensions add up to +5 bonus (capped at 100) only if mandatory scope is ≥ 80.**

> **A correct, well-justified null or non-identifiability verdict earns full marks on the
> relevant criteria.** The graders read *reasoning and honesty first*, point estimate second.

## Anti-patterns that cap a criterion at 70%

- metrics chosen after seeing data
- peeking-driven stopping
- controlling for a mediator or collider
- presenting an observational estimate with no assumption or refutation
- reporting bare p-values to leadership

## Live verification

At submission the instructor will ask you, out loud:

> *"State the identifying assumption of your causal estimate, and the single unmeasured
> variable that would break it."*

Section 8 exists so that you have that sentence ready.

---

### Data available to you

| File | Rows | What it is for |
|---|---|---|
| `injaz_autofill_ab.csv` | 70,000 | the autofill randomised experiment |
| `injaz_traffic_forecast.csv` | 90 | daily eligible traffic for run-length planning |
| `injaz_pre_period.csv` | 50,000 | pre-experiment window for CUPED |
| `injaz_regions_panel.csv` | 312 | the staggered reminder rollout (13 regions × 24 months) |
| `injaz_regions_panel_pretrend.csv` | 312 | the same, with parallel trends broken |
| `injaz_users.csv` | 20,000 | observational cohort: training, channel choice, the IV |
| `injaz_users_pool.csv` | 200,000 | the eligible pool, for assignment work |
| `injaz_sessions.csv.gz` | 339,017 | session-level events (clustering demonstrations) |
| `injaz_experiment_results.csv` | 80,000 | the guided-uploader experiment |
| `collider_sim.parquet` | 50,000 | the collider demonstration |

`data/DATA_DICTIONARY.md` describes every column.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `causal_utils` importable from anywhere under course_assets/
ROOT = Path.cwd()
while not (ROOT / "causal_utils").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from causal_utils import use_sdaia_style
use_sdaia_style()

RANDOM_SEED = 213          # fixed for every lab: reproducibility is graded
rng = np.random.default_rng(RANDOM_SEED)
print("Environment ready ·", ROOT)

In [ ]:
# --- lab scaffolding: self-checks that never crash the notebook ----------
def check(name, fn):
    """Run a self-check. Prints PASS/FAIL/TODO instead of raising, so the
    notebook always runs top-to-bottom even with unfinished exercises."""
    try:
        fn()
    except AssertionError as e:
        print(f"[ TODO ]  {name}: {e}")
        return False
    except Exception as e:
        print(f"[ ERROR ] {name}: {type(e).__name__}: {e}")
        return False
    print(f"[  OK  ]  {name}")
    return True


def needs(*objs, msg="complete the TODO cell above first"):
    """Guard downstream cells so an unfinished TODO skips rather than crashes."""
    if any(o is None for o in objs):
        print(f"[ SKIPPED ] {msg}")
        return False
    return True

print("Scaffolding loaded: use check(...) to self-test and needs(...) to guard.")

In [ ]:
# ---------------------------------------------------------------------------
# Everything you need is in causal_utils. Import what you use; do not reinvent.
# ---------------------------------------------------------------------------
from causal_utils import (
    # assignment & integrity
    hash_assign, srm_check, balance_table, standardised_mean_difference,
    # power & variance reduction
    sample_size_proportions, sample_size_means, power_for_n, mde_for_n,
    run_length, cuped_adjust,
    # analysis
    two_proportion_test, regression_effect, delta_method_ratio,
    correct_pvalues, always_valid_bound, practical_significance_verdict,
    peeking_simulation,
    # quasi-experimental
    estimate_propensity, match_nearest, ipw_ate, did_twfe, event_study, iv_2sls,
    rosenbaum_bounds, placebo_outcome_test, random_common_cause_test,
    subset_refutation,
    # graphs
    InjazDAG, backdoor_adjustment_sets, is_valid_adjustment_set, classify_triple,
    draw_dag,
    # reporting
    AnalysisPlan, freeze_plan, decision_plot, forest_plot, decision_memo,
    # palette
    NAVY, BLUE, ORANGE, PURPLE, TEAL, SLATE, MUTED, PANEL,
)
import networkx as nx
import statsmodels.api as sm
import statsmodels.formula.api as smf

print("causal_utils loaded. RANDOM_SEED =", RANDOM_SEED)
print("Reproducibility is graded: do not change the seed, and do not add "
      "un-seeded randomness.")

---

# §1 — Frame the decisions

*Before any code that touches an outcome.*

Write, in business language, what is being decided and what quantity would decide it.
"Compared to what, for whom?" is the whole of framing.

In [ ]:
# TODO: fill in every field. These are strings; write them for a director, not a
#       statistician. `estimand` must name BOTH the quantity (ATE / ATT / LATE) and the
#       POPULATION it applies to.

FRAME = {
    "decision_1": "TODO - the autofill launch decision, in one sentence",
    "decision_2": "TODO - the reminder scaling decision, in one sentence",
    "estimand_autofill": "TODO - e.g. 'ATE on completion among users who trigger autofill'",
    "estimand_reminders": "TODO - e.g. 'ATT on completion rate in activated regions'",
    "why_ab_for_autofill": "TODO - one sentence on why randomisation is available here",
    "why_not_ab_for_reminders": "TODO - one sentence on why it is not available there",
    "decision_threshold_pp": None,   # TODO - the completion gain that justifies the build
    "who_decides": "TODO - the forum and the date",
}

def _c():
    assert all(v not in (None, "") and not str(v).startswith("TODO")
               for v in FRAME.values()), "some FRAME fields are still TODO"
    assert isinstance(FRAME["decision_threshold_pp"], (int, float)), \
        "decision_threshold_pp must be a number, in percentage points"
check("§1 framing complete", _c)

for k, v in FRAME.items():
    print(f"{k:26s} {v}")

---

# §2 — Design doc and frozen analysis plan  *(rubric: 15 pts)*

**90–100% band:** unit justified, OEC + guardrails sound, plan frozen *before* results.
**< 70%:** no plan, or metrics chosen after seeing data.

The `plan_hash` printed by `freeze_plan()` is your process evidence. Put it in the memo.

Required content:
- randomisation unit **and a justification** for it
- the OEC
- **at least three** guardrails
- the trigger definition
- the unit of analysis (and whether it matches the randomisation unit)
- the MDE and the practical-significance threshold
- the multiple-testing correction family
- the stopping rule

In [ ]:
# TODO: complete the AnalysisPlan and freeze it. Every field below is graded.
#       Read the docstring: `AnalysisPlan` names what must be committed before outcomes.

plan = AnalysisPlan(
    experiment_name="TODO",
    decision="TODO",
    randomisation_unit="TODO",
    oec="TODO",
    guardrails=[],                    # TODO - at least three
    trigger_definition="TODO",
    unit_of_analysis="TODO",
    mde=None,                         # TODO - as a proportion, e.g. 0.010 for 1.0 pp
    alpha=0.05,
    power=0.80,
    correction_family="TODO",
    stopping_rule="TODO",
    practical_threshold=None,         # TODO - as a proportion
)

# TODO: uncomment once the fields above are filled in
# plan = freeze_plan(plan)

def _c():
    assert plan.mde is not None and plan.practical_threshold is not None, \
        "mde and practical_threshold must be set"
    assert len(plan.guardrails) >= 3, "at least three guardrails are required"
    assert not any(str(getattr(plan, f)).startswith("TODO")
                   for f in ("experiment_name", "decision", "randomisation_unit", "oec",
                             "trigger_definition", "unit_of_analysis",
                             "correction_family", "stopping_rule")), \
        "some plan fields are still TODO"
    assert plan.plan_hash is not None, "you must call freeze_plan(plan) - the freeze is graded"
check("§2 analysis plan complete and FROZEN", _c)

In [ ]:
# TODO: justify the randomisation unit in RANDOMISATION_JUSTIFICATION (2-3 sentences).
#   Address: (a) does a user see the feature consistently across sessions?
#            (b) is there interference between units?
#            (c) does the unit of analysis match the randomisation unit - and if not,
#                what does that do to the standard errors?

RANDOMISATION_JUSTIFICATION = """TODO"""
print(RANDOMISATION_JUSTIFICATION)

def _c():
    assert not RANDOMISATION_JUSTIFICATION.strip().startswith("TODO"), "still TODO"
    assert len(RANDOMISATION_JUSTIFICATION.split()) >= 40, \
        "two or three real sentences, please - this is 15 points"
check("§2 randomisation unit justified", _c)

In [ ]:
# --- process evidence: version the plan BEFORE any results ------------------
# TODO: write the frozen plan to disk so its timestamp precedes your results.
#       Uncomment once `plan` is frozen.
#
# PLAN_PATH = Path("capstone_analysis_plan.json")
# PLAN_PATH.write_text(plan.to_json())
# print(f"plan written to {PLAN_PATH.resolve()}")
# print(f"hash {plan.plan_hash} - quote this in the memo")

print("[ TODO ] version the frozen plan to disk (uncomment the block above)")

---

# §3 — Power and sample sizing  *(rubric: 15 pts)*

**90–100% band:** correct sizing including the **arm multiplier**, the **trigger rate** and
the **weekly floor**; CUPED saving quantified.
**< 70%:** wrong or missing sizing; an under-run experiment not recognised as under-run.

Four things must appear:

1. required **n per arm** at your MDE
2. the **run-length plan** — arms × trigger rate × whole weeks
3. the **CUPED-adjusted** alternative and its quantified saving
4. what you *needed* against what you *have*: `mde_for_n(n_actual, baseline)` — and an
   explicit statement of whether this run can resolve its own decision threshold

In [ ]:
ab = pd.read_csv(DATA / "injaz_autofill_ab.csv")
traffic = pd.read_csv(DATA / "injaz_traffic_forecast.csv")
pre = pd.read_csv(DATA / "injaz_pre_period.csv")

print(f"autofill experiment : {len(ab):,} assigned · "
      f"trigger rate {ab['triggered'].mean():.1%}")
print(f"traffic forecast    : {len(traffic)} days · "
      f"mean {traffic['eligible_users'].mean():,.0f} eligible/day")
ab.head(3)

In [ ]:
# TODO: sizing.
#   baseline    - completion rate among TRIGGERED CONTROL users
#   n_required  - sample_size_proportions(baseline, plan.mde, alpha=..., power=...)
#   rl          - run_length(n_required, daily_eligible=..., trigger_rate=...)
#   cuped       - cuped_adjust(y, x_pre) on the triggered population
#   n_cuped     - the CUPED-adjusted requirement, and the % saving
#   n_actual    - the smaller arm's triggered sample
#   mde_actual  - mde_for_n(n_actual, baseline)

triggered = ab[ab["triggered"] == 1].copy()

baseline = None
n_required = None
rl = None
cuped = None
n_cuped = None
n_actual = None
mde_actual = None

def _c():
    for name in ("baseline", "n_required", "rl", "cuped", "n_cuped", "n_actual",
                 "mde_actual"):
        assert globals()[name] is not None, f"{name} is still None"
    assert 0.4 < baseline < 0.7, "baseline completion looks wrong"
    assert n_cuped < n_required, "CUPED should REDUCE the required sample"
check("§3 sizing, run length and CUPED saving computed", _c)

if needs(baseline, n_required, rl, cuped, n_cuped, n_actual, mde_actual):
    print(rl)
    print(cuped)
    print(f"\nCUPED saving: {100*(1 - n_cuped/n_required):.1f}% of the sample")
    print(f"needed per arm {n_required:,} vs collected {n_actual:,}")
    print(f"smallest resolvable effect {100*mde_actual:+.2f} pp vs "
          f"threshold {100*plan.practical_threshold:+.2f} pp")

In [ ]:
# TODO: SIZING_VERDICT - two or three sentences.
#   Does this run have the power to answer the decision question? Say so explicitly,
#   in the language of the threshold, not of p-values. If it does not, that is a
#   FINDING you will carry into §10, not a problem to hide.

SIZING_VERDICT = """TODO"""
print(SIZING_VERDICT)

def _c():
    assert not SIZING_VERDICT.strip().startswith("TODO"), "still TODO"
check("§3 sizing verdict written", _c)

---

# §4 — Experiment integrity  *(rubric: 10 pts)*

**90–100% band:** SRM + balance correct; an invalid experiment is flagged as invalid.
**< 70%:** no SRM or balance; an invalid test analysed anyway.

Three artefacts:

1. **Deterministic hash assignment** — demonstrate `hash_assign` reproduces stable arms from
   the unit id alone.
2. **SRM check** — on the assigned population **and** on the triggered subset.
3. **Covariate balance table** — every |SMD| < 0.1, on pre-treatment covariates only.

In [ ]:
# TODO: demonstrate deterministic assignment.
#   Run hash_assign on injaz_users_pool.csv user ids with your own salt, twice,
#   and show the two runs are identical. Then show the realised allocation.

pool = pd.read_csv(DATA / "injaz_users_pool.csv", usecols=["user_id", "is_eligible"])
eligible = pool[pool["is_eligible"] == 1]

assign_a = None    # TODO
assign_b = None    # TODO

def _c():
    assert assign_a is not None and assign_b is not None, "run hash_assign twice"
    assert (np.asarray(assign_a) == np.asarray(assign_b)).all(), \
        "hash assignment must be deterministic - same salt, same ids, same arms"
check("§4 assignment is deterministic and reproducible", _c)

In [ ]:
# TODO: srm_check on the assigned population AND on the triggered subset,
#       then balance_table on PRE-TREATMENT covariates within the triggered subset.
#       Choose the covariates yourself - and be able to say why each is pre-treatment.

BALANCE_COVS = []      # TODO
srm_all = None
srm_trig = None
bal = None

def _c():
    assert srm_all is not None and srm_trig is not None and bal is not None, \
        "run all three checks"
    assert len(BALANCE_COVS) >= 4, "check at least four covariates"
    assert "completed" not in BALANCE_COVS and "support_contact" not in BALANCE_COVS, \
        "those are OUTCOMES - balance is checked on PRE-treatment covariates only"
    assert srm_all.passed and srm_trig.passed, \
        "SRM failed: the experiment is INVALID and must not be analysed - say so"
    assert bal["abs_smd"].max() < 0.10, "a covariate is imbalanced"
check("§4 SRM passes and every |SMD| < 0.1", _c)

if needs(srm_all, srm_trig, bal):
    print(srm_all); print(); print(srm_trig); print()
    print(bal.head(8).to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

In [ ]:
# TODO: INTEGRITY_STATEMENT - one paragraph.
#   What did you check, what passed, and what would you have done had SRM failed?
#   (The correct answer to a failed SRM is not "adjust and continue".)

INTEGRITY_STATEMENT = """TODO"""
print(INTEGRITY_STATEMENT)

def _c():
    assert not INTEGRITY_STATEMENT.strip().startswith("TODO"), "still TODO"
check("§4 integrity statement written", _c)

---

# §5 — A/B analysis: the OEC  *(rubric: 20 pts, with §6)*

**90–100% band:** robust or clustered variance, CUPED, correct correction family, and a
clear **statistical-vs-practical** verdict against the design threshold.
**< 70%:** naive t-test, no correction, p-value worship.

Required:

1. the effect on the **frozen OEC**, on the **triggered** population
2. **CUPED-adjusted**, with robust (HC3) standard errors
3. the interval, in percentage points
4. the **dilution demonstration**: what an all-assigned analysis would have said, and why
   that is the wrong denominator

In [ ]:
# TODO: estimate the OEC effect three ways.
#   eff_raw   - two_proportion_test on triggered users
#   eff_cuped - regression_effect on the CUPED-adjusted outcome, cov_type="HC3"
#   eff_all   - two_proportion_test on ALL assigned users (the dilution demo)

eff_raw = None
eff_cuped = None
eff_all = None

def _c():
    assert eff_raw is not None and eff_cuped is not None and eff_all is not None, \
        "produce all three estimates"
    assert eff_cuped.se < eff_raw.se, \
        "CUPED should tighten the interval - check you used a PRE-treatment covariate"
check("§5 OEC estimated; CUPED tightens the interval", _c)

if needs(eff_raw, eff_cuped, eff_all):
    for e in (eff_raw, eff_cuped, eff_all):
        print(e); print()

In [ ]:
# TODO: CUPED_SAFETY - one sentence naming the covariate you used and stating why it
#       could not have been affected by treatment. This is the entire safety argument
#       for CUPED and graders look for it.

CUPED_SAFETY = """TODO"""
print(CUPED_SAFETY)

def _c():
    assert not CUPED_SAFETY.strip().startswith("TODO"), "still TODO"
check("§5 CUPED safety argument stated", _c)

---

# §6 — Guardrails and the correction family  *(rubric: 20 pts, with §5)*

The correction family was a **design** decision, frozen in §2. Apply it — do not choose it
now.

| Family | Correction | Why |
|---|---|---|
| one pre-registered OEC | none | a single planned test |
| a guardrail family (3–6) | **Bonferroni** (FWER) | you want to be sure *none* is broken |
| an exploratory segment sweep | **BH** (FDR) | some false discoveries are tolerable |

Report **direction** as well as significance — a guardrail that moves the *right* way is good
news, not a failure.

In [ ]:
# TODO: estimate each pre-registered guardrail with regression_effect (HC3),
#       then apply correct_pvalues with the family and method from your frozen plan.
#       Add a `direction` column saying which way is harm.

guardrail_effects = None
guardrails = None

def _c():
    assert guardrails is not None, "guardrails is still None"
    assert len(guardrails) == len(plan.guardrails), \
        "one row per PRE-REGISTERED guardrail - no adding or dropping after the fact"
check("§6 guardrails checked with the pre-registered family", _c)

if needs(guardrails):
    print(guardrails.to_string(index=False))

In [ ]:
# TODO: GUARDRAIL_VERDICT - one short paragraph.
#   Is the feature SAFE? Name any metric that moved, whether it survived correction,
#   and which direction it moved in. Distinguish "safe" from "worth shipping".

GUARDRAIL_VERDICT = """TODO"""
print(GUARDRAIL_VERDICT)

def _c():
    assert not GUARDRAIL_VERDICT.strip().startswith("TODO"), "still TODO"
check("§6 guardrail verdict written", _c)

---

# §7 — Causal evaluation of the reminder pilot  *(rubric: 20 pts, with §8)*

**90–100% band:** DAG + assumption stated; DiD or matching valid; **≥ 2 refutations**;
sensitivity honest.
**< 70%:** no identification argument; an unvalidated causal claim.

Choose **one** route and do it properly:

- **DiD with an event-study parallel-trends check** on `injaz_regions_panel.csv`, or
- **Matching with sensitivity analysis** on `injaz_users.csv`

Then run **at least two** validation / refutation checks. For DiD those are the event study,
a placebo-timing test, and a clean not-yet-treated comparison. For matching they are the
placebo outcome, random common cause, data subset, and Rosenbaum bounds.

In [ ]:
# TODO: state your route and your identifying assumption BEFORE estimating.
CAUSAL_ROUTE = "TODO - 'did' or 'matching'"
IDENTIFYING_ASSUMPTION = """TODO - one sentence, in words a non-statistician can attack"""
CREDIBLE_THREAT = """TODO - the single most plausible way this assumption fails"""

def _c():
    assert CAUSAL_ROUTE in ("did", "matching"), "choose 'did' or 'matching'"
    for s in (IDENTIFYING_ASSUMPTION, CREDIBLE_THREAT):
        assert not s.strip().startswith("TODO"), "still TODO"
check("§7 route chosen and identifying assumption stated FIRST", _c)

print("ROUTE                 :", CAUSAL_ROUTE)
print("IDENTIFYING ASSUMPTION:", IDENTIFYING_ASSUMPTION)
print("CREDIBLE THREAT       :", CREDIBLE_THREAT)

In [ ]:
# TODO: estimate the pilot effect on your chosen route.
#   DiD route      -> did_twfe(...) then event_study(...) and plot it
#   Matching route -> estimate_propensity / match_nearest / ipw_ate, plus the
#                     overlap plot and the post-matching balance table
#   Store the headline estimate object in `causal_estimate`.

causal_estimate = None

def _c():
    assert causal_estimate is not None, "causal_estimate is still None"
    for attr in ("ci_low", "ci_high"):
        assert hasattr(causal_estimate, attr), \
            "your estimate object must carry a confidence interval"
check("§7 pilot effect estimated with an interval", _c)

if needs(causal_estimate):
    print(causal_estimate)

In [ ]:
# TODO: the credibility exhibit.
#   DiD route      -> the event-study plot: flat pre-period, jump at activation, CIs
#   Matching route -> the love plot: |SMD| before vs after, with the 0.10 line
#   Every chart needs a title, axis labels in percentage points, and an annotation
#   stating the takeaway.

print("[ TODO ] credibility exhibit")

In [ ]:
# TODO: at least TWO refutation / validation checks. Store their results in
#       `validation` (a dict) so §11's self-audit can count them.

validation = None

def _c():
    assert validation is not None, "validation is still None"
    assert len(validation) >= 2, "the rubric requires at least two refutation checks"
check("§7 at least two refutation / validation checks run", _c)

if needs(validation):
    for k, v in validation.items():
        print(f"{k:32s} {v}")

---

# §8 — The DAG and the identification verdict  *(rubric: 20 pts, with §7)*

Draw the graph for your causal question, derive the adjustment set from it, and issue **one
of three verdicts**:

1. **identifiable by backdoor set S**
2. **identifiable by another route** (frontdoor / IV)
3. **not identifiable with current data** — and here is exactly what is missing

Then prepare the sentence the instructor will ask you for out loud.

In [ ]:
# TODO: build the DAG for your causal question, derive the adjustment set, and
#       confirm no descendant of the treatment is in it.

G = None
dag = None
adjustment_set = None

def _c():
    assert G is not None and dag is not None, "G / dag still None"
    assert nx.is_directed_acyclic_graph(G), "your graph has a cycle"
    assert adjustment_set is not None, "derive the adjustment set from the graph"
    desc = set(nx.descendants(G, dag.treatment))
    assert not (set(adjustment_set) & desc), \
        "your adjustment set contains a DESCENDANT of the treatment (mediator/collider)"
check("§8 DAG built and adjustment set derived, mediators excluded", _c)

if needs(dag):
    print(dag.explain())

In [ ]:
# TODO: draw the DAG with draw_dag(...), colouring the adjustment set.
print("[ TODO ] DAG figure")

In [ ]:
# TODO: the identification verdict, and the sentence for the live check.

VERDICT_TYPE = None            # TODO: 1, 2 or 3
IDENTIFICATION_VERDICT = """TODO - the full verdict paragraph"""
LIVE_CHECK_SENTENCE = """TODO - 'My identifying assumption is ___. The single unmeasured
variable that would break it is ___.'"""

def _c():
    assert VERDICT_TYPE in (1, 2, 3), "the verdict must be 1, 2 or 3"
    for s in (IDENTIFICATION_VERDICT, LIVE_CHECK_SENTENCE):
        assert not s.strip().startswith("TODO"), "still TODO"
    assert len(LIVE_CHECK_SENTENCE.split()) >= 20, "say it properly - you will be asked"
check("§8 identification verdict issued", _c)

print(IDENTIFICATION_VERDICT)
print("\nLIVE CHECK:\n", LIVE_CHECK_SENTENCE)

---

# §9 — Reconcile and decide  *(rubric: 15 pts, with §10)*

One axis, one threshold line, every estimate with its interval. Then the verdict — the
**interval** against the **threshold**, never the p-value against zero.

Remember the four cases and that only one of them is *ship*.

In [ ]:
# TODO: build the decision plot with decision_plot(estimates, threshold=...).
#       Include the autofill A/B effect and your pilot estimate at minimum.
#       Annotate the takeaway on the figure.

ESTIMATES = None

def _c():
    assert ESTIMATES is not None, "ESTIMATES is still None"
    assert len(ESTIMATES) >= 2, "reconcile at least the A/B and the pilot estimate"
check("§9 decision plot built", _c)

In [ ]:
# TODO: the practical-significance verdict for the autofill decision.
#       Use practical_significance_verdict(eff, threshold, name) and then WRITE OUT
#       the reasoning: p-value vs interval vs threshold, in your own words.

DECISION_REASONING = """TODO"""

if needs(eff_cuped):
    print(practical_significance_verdict(eff_cuped, plan.practical_threshold,
                                         "Autofill completion"))
print("\n" + DECISION_REASONING)

def _c():
    assert not DECISION_REASONING.strip().startswith("TODO"), "still TODO"
    assert len(DECISION_REASONING.split()) >= 60, \
        "this is the graded judgement - spell out the reasoning"
check("§9 decision reasoning written", _c)

In [ ]:
# TODO: reconcile the methods. If your A/B and quasi-experimental estimates
#       disagree, explain the gap via the likely violated assumption. Do NOT
#       cherry-pick. Rank the interventions by STRENGTH OF EVIDENCE, which is
#       often the opposite order to effect size.

RECONCILIATION = """TODO"""
print(RECONCILIATION)

def _c():
    assert not RECONCILIATION.strip().startswith("TODO"), "still TODO"
check("§9 methods reconciled, not cherry-picked", _c)

---

# §10 — The one-page decision memo  *(rubric: 15 pts, with §9)*

**90–100% band:** effects **and intervals in business units**, a clear recommendation per
intervention, key assumptions stated **as risks**, and an explicit **"what would change
this."**
**< 70%:** p-values to leadership; overstated certainty.

**The memo is the deliverable. The notebook is the evidence behind it.**

No p-values reach this page. A director cannot act on 0.0001; they can act on "between +0.8
and +2.4 points, and we need at least +1.0."

In [ ]:
# TODO: render the memo with decision_memo(...). Pass `plan=plan` so the frozen-plan
#       hash appears at the top as process evidence.

memo = None

def _c():
    assert memo is not None, "memo is still None"
    assert plan.plan_hash in memo, "pass plan=plan so the frozen hash appears in the memo"
    lowered = memo.lower()
    assert "p =" not in lowered and "p-value" not in lowered and "p<" not in lowered, \
        "no p-values in a decision memo"
    for word in ("ship", "scale", "hold", "stop", "expand"):
        if word in lowered:
            break
    else:
        raise AssertionError("the memo must contain a clear recommendation verb")
    assert "what would change this" in lowered, \
        "an explicit 'what would change this' section is required"
check("§10 memo: intervals in business units, recommendation, risks, no p-values", _c)

if needs(memo):
    print(memo)

In [ ]:
# TODO: save the memo as a standalone artefact for submission.
# MEMO_PATH = Path("capstone_decision_memo.txt")
# MEMO_PATH.write_text(memo)
# print("memo written to", MEMO_PATH.resolve())

print("[ TODO ] write the memo to disk (uncomment above)")

---

# §11 — Extension  *(choose at least one; +5 bonus, capped at 100, if mandatory scope ≥ 80)*

| # | Extension | Where to start |
|---|---|---|
| 1 | **Heterogeneous treatment effects (CATE)** on the autofill test, with an honest note on multiple testing across subgroups | segment the triggered population, `regression_effect` per segment, `correct_pvalues(..., method="bh")` |
| 2 | **Always-valid / sequential inference** enabling legitimate continuous monitoring | `always_valid_bound(n)` and `peeking_simulation()` |
| 3 | **Staggered-DiD estimator** (Callaway–Sant'Anna / Sun–Abraham) on the full multi-wave rollout, compared against naive TWFE | build cohort-by-cohort 2×2s against never-treated regions and aggregate |
| 4 | **Instrumental variables** for filing channel, with a defended instrument and first-stage F | `iv_2sls(users, "completed", "online_filing", "log_distance_km", exog=[...])` — use `log_distance_km`, not raw km |
| 5 | **Power / variance-reduction study** quantifying how much CUPED and stratification would shorten future Injaz experiments | `cuped_adjust`, `sample_size_proportions`, `run_length` across a grid of MDEs |

> **EconML and DoWhy are not installed.** Extension 1 does not need a causal forest: honest
> subgroup estimates with a BH correction and a clear statement about multiplicity score full
> marks. Extension 3 does not need a package: the cohort-by-cohort 2×2 construction *is* the
> Callaway–Sant'Anna idea.

In [ ]:
# TODO: implement your chosen extension here.
#       Name it, implement it, and state in one sentence what it added that the
#       mandatory scope did not.

EXTENSION_NAME = "TODO"
EXTENSION_FINDING = """TODO - what did this add?"""

# ... your extension code ...

def _c():
    assert not EXTENSION_NAME.startswith("TODO"), "name your extension"
    assert not EXTENSION_FINDING.strip().startswith("TODO"), "state what it added"
check("§11 extension implemented", _c)

print(EXTENSION_NAME)
print(EXTENSION_FINDING)

---

# §12 — Self-audit checklist

Run the cell below **before you submit**. It walks the rubric and reports what is missing.
It is not a grader — it checks that the required artefacts exist, not that they are good.

Anything it flags is a point you have not yet earned.

In [ ]:
def self_audit():
    """Walk the rubric and report which required artefacts are present."""
    g = globals()

    def ok(cond):
        return "PASS" if cond else "MISSING"

    def filled(name):
        v = g.get(name)
        return v is not None and not str(v).strip().startswith("TODO")

    def has_ci(name):
        v = g.get(name)
        return v is not None and hasattr(v, "ci_low") and hasattr(v, "ci_high")

    rows = [
        # criterion, weight, item, condition
        ("Design & plan", 15, "FRAME complete",
         all(filled_v not in (None, "") and not str(filled_v).startswith("TODO")
             for filled_v in g.get("FRAME", {"x": "TODO"}).values())),
        ("Design & plan", 15, "AnalysisPlan frozen (hash exists)",
         g.get("plan") is not None and getattr(g["plan"], "plan_hash", None) is not None),
        ("Design & plan", 15, ">= 3 guardrails pre-registered",
         g.get("plan") is not None and len(getattr(g["plan"], "guardrails", [])) >= 3),
        ("Design & plan", 15, "randomisation unit justified",
         filled("RANDOMISATION_JUSTIFICATION")),

        ("Power & sizing", 15, "n per arm computed", filled("n_required")),
        ("Power & sizing", 15, "run length incl. arms/trigger/weekly floor", filled("rl")),
        ("Power & sizing", 15, "CUPED saving quantified",
         filled("cuped") and filled("n_cuped")),
        ("Power & sizing", 15, "achievable MDE vs threshold stated",
         filled("mde_actual") and filled("SIZING_VERDICT")),

        ("Integrity", 10, "deterministic assignment demonstrated", filled("assign_a")),
        ("Integrity", 10, "SRM on assigned AND triggered",
         filled("srm_all") and filled("srm_trig")),
        ("Integrity", 10, "balance table, all |SMD| < 0.1",
         g.get("bal") is not None and float(g["bal"]["abs_smd"].max()) < 0.10),
        ("Integrity", 10, "integrity statement written", filled("INTEGRITY_STATEMENT")),

        ("A/B analysis", 20, "OEC effect with robust variance", has_ci("eff_cuped")),
        ("A/B analysis", 20, "CUPED applied and its safety argued",
         filled("cuped") and filled("CUPED_SAFETY")),
        ("A/B analysis", 20, "dilution / triggered-population handled", has_ci("eff_all")),
        ("A/B analysis", 20, "guardrails with the pre-registered correction",
         filled("guardrails") and filled("GUARDRAIL_VERDICT")),

        ("Causal evaluation", 20, "identifying assumption stated FIRST",
         filled("IDENTIFYING_ASSUMPTION") and filled("CREDIBLE_THREAT")),
        ("Causal evaluation", 20, "pilot effect with an interval", has_ci("causal_estimate")),
        ("Causal evaluation", 20, ">= 2 refutation / validation checks",
         g.get("validation") is not None and len(g["validation"]) >= 2),
        ("Causal evaluation", 20, "DAG built, no descendant of T in the adjustment set",
         g.get("G") is not None and g.get("adjustment_set") is not None),
        ("Causal evaluation", 20, "identification verdict issued",
         g.get("VERDICT_TYPE") in (1, 2, 3) and filled("IDENTIFICATION_VERDICT")),

        ("Communication", 15, "decision plot vs threshold", filled("ESTIMATES")),
        ("Communication", 15, "practical-vs-statistical reasoning written",
         filled("DECISION_REASONING")),
        ("Communication", 15, "methods reconciled, not cherry-picked",
         filled("RECONCILIATION")),
        ("Communication", 15, "memo rendered with the plan hash",
         g.get("memo") is not None and g.get("plan") is not None
         and getattr(g["plan"], "plan_hash", "") in str(g.get("memo", ""))),
        ("Communication", 15, "'what would change this' present",
         "what would change this" in str(g.get("memo", "")).lower()),

        ("Reproducibility", 5, "fixed seed untouched", g.get("RANDOM_SEED") == 213),
        ("Reproducibility", 5, "plan frozen before results (hash on disk)",
         Path("capstone_analysis_plan.json").exists()),

        ("Extension (+5)", 0, "extension implemented",
         filled("EXTENSION_NAME") and filled("EXTENSION_FINDING")),
    ]

    out = pd.DataFrame([{"criterion": c, "weight": w, "item": i, "status": ok(cond)}
                        for c, w, i, cond in rows])
    print(out.to_string(index=False))

    missing = out[out["status"] == "MISSING"]
    print("\n" + "=" * 72)
    if missing.empty:
        print("SELF-AUDIT: every required artefact is present.")
        print("That is necessary, not sufficient. The graders read reasoning and honesty")
        print("first, the point estimate second. Re-read your memo as a director would:")
        print("  - is every number given with a range, in business units?")
        print("  - is the recommendation compared to the THRESHOLD, not to zero?")
        print("  - are the assumptions stated as risks the reader can act on?")
        print("  - could a reviewer tell, from this notebook alone, that the metrics")
        print("    were chosen before the results?")
    else:
        print(f"SELF-AUDIT: {len(missing)} required artefact(s) still missing.")
        for _, r in missing.iterrows():
            print(f"  [{r['criterion']:<18} {r['weight']:>2} pts]  {r['item']}")
    print("=" * 72)
    return out


audit = self_audit()

---

## §13 — Submission

1. **This notebook**, run clean top-to-bottom, seed untouched.
2. **`capstone_analysis_plan.json`** — the frozen plan, timestamped *before* your results.
3. **`capstone_decision_memo.txt`** — the one-page memo.
4. **A 6-minute presentation**: the two decisions, the evidence and its limits, and the
   recommendation.

### Before you present, rehearse these three answers

1. *"What is your identifying assumption, and what single unmeasured variable would break
   it?"*
2. *"Your confidence interval crosses the decision threshold. What do you recommend, and
   why is that not indecision?"*
3. *"The board wants one number. Give it to them honestly."*

### A closing note

A correct, well-justified **null** or **non-identifiability** verdict earns full marks. A
confident wrong answer is a failure; an honest *"not yet, and here is what it would take"* is
a service.

That distinction is the whole course.

---

*SDA-DSC-213 · Experimentation, A/B Testing and Causal Inference · SDAIA Academy*
*جميع البيانات المستخدمة في هذه الدورة اصطناعية بالكامل — لا تُستخدم أي بيانات حقيقية للمواطنين أو الجهات الحكومية.*